# 2 · One question through both loops — live, a fraction of a cent

**This notebook spends money.** It needs `OPENAI_API_KEY` and makes a handful of
calls. Everything in notebook 1 is free; nothing here is.

It runs one question twice: once through Level 1 (retry on execution failure) and once
through Level 2 (the parse-tree verifiers wrapped around it), and shows what each
loop could and could not see.

> Imports from `loopeng`, no loop logic. The retry policy, the budget check and the
> verifier feedback all live in `src/loopeng/`; this notebook calls them and prints
> what came back.

In [ ]:
import os
from pathlib import Path

# Jupyter starts the kernel in the notebook's own directory, and
# everything in this repository is addressed from the root: `.env`,
# the warehouse, `gold/`, `results/`. Move there once. Idempotent, so
# re-running the cell is a no-op.
if not Path("pyproject.toml").exists():
    os.chdir("..")
print("working from", Path.cwd().name)


In [ ]:
from pathlib import Path

from loopeng.gold.build import read_gold, split_items
from loopeng.settings import load_settings
from loopeng.warehouse.connect import ensure_warehouse

settings = load_settings()
warehouse = ensure_warehouse(Path("..") / settings.warehouse_path,
                             seed=settings.warehouse_seed)
held_out, _ = split_items(read_gold(Path("gold/gold.jsonl")))
print("warehouse and gold set ready")

## Pick a question the rules actually bite on

An item with no applicable rules cannot show the difference between the levels, so
choose one that has some.

In [ ]:
item = next(i for i in held_out if len(i.rules) >= 2)
print(item.question)
print("\nrules that apply:", ", ".join(item.rules))

## Level 1: retry when the query fails to EXECUTE

This loop sees crashes. It cannot see a query that ran cleanly and returned the wrong
number — which is the category the whole session is about.

In [ ]:
from loopeng.agent.loop import run_question

l1 = run_question(item.question, warehouse=warehouse, level="L3",
                  max_attempts=3, item_id=item.item_id)

print(f"termination : {l1.termination}")
print(f"attempts    : {len(l1.attempts)}")
print(f"cost        : est. ${l1.cost_usd():.6f}")
print(f"served model: {l1.served_model}")

In [ ]:
for attempt in l1.attempts:
    print(f"--- attempt {attempt.n} ---")
    print(attempt.sql)
    print("error:", attempt.error or "none")
    print("rows :", attempt.rows)
    print()

## Level 2: the same generator, with verifiers around it

The verifiers read the **parsed query**, not its text, and reject it for breaking a
declared rule even when it ran perfectly well. `rejections` counts how many times that
happened.

**The verifier cannot see the gold answer.** That is structural rather than a
convention: `VerifyContext` has no field that could carry it, and a test asserts no
field name could.

In [ ]:
from loopeng.verify.loop import run_verified

l2 = run_verified(item.question, warehouse=warehouse, rules=item.rules, level="L3",
                  max_attempts=3, item_id=item.item_id)

print(f"termination : {l2.termination}")
print(f"attempts    : {len(l2.attempts)}")
print(f"rejections  : {l2.rejections}")
print(f"cost        : est. ${l2.cost_usd():.6f}")

In [ ]:
print(l2.sql)
print()
print("rows:", l2.rows)

## What the judge says about each

`judge` compares the answer against the gold rows. The outcome is one of a fixed set,
and every one of them maps to a band — `band_of` raises on an unmapped outcome rather
than defaulting, because a new outcome silently joining the wrong band is how a right
answer once got counted as a wrong one.

The Level 2 run goes through `as_agent_run` first. Writing this notebook without it
raised `AttributeError: 'VerifiedAttempt' object has no attribute
'model_call_failed'` — which is the guard rail working: one judge, one shape, and a
loud failure rather than two scoring paths that agree until they do not.


In [ ]:
from loopeng.agent.classify import band_of, judge
from loopeng.verify.batch import as_agent_run

# `as_agent_run` is not a formality. A verified run carries the verifier's rejections
# and its own attempt type; the adapter puts it in the shape ONE judge scores, so both
# arms are graded by the same code rather than by two that agree today.
for name, run in (("L1", l1), ("L2", as_agent_run(l2))):
    verdict = judge(run, item)
    print(f"{name}: {verdict.outcome}  ->  band {band_of(verdict.outcome)}")


### The distinction to take away

A **visible failure** is detectably wrong without knowing the answer: invalid SQL, a
timeout, the wrong shape. A retry loop can see those.

A **silent error** ran, returned one plausible number, and is wrong. Nothing in the
output distinguishes it from a correct answer. Verification exists to convert the
second into the first — it does not mainly make an agent right, it makes it stop being
confidently wrong, and those move independently.

In [ ]:
print(f"est. ${l1.cost_usd() + l2.cost_usd():.6f} spent in this notebook")